    # Week 8 · Python Vector Workflows
    
    This notebook introduces the managed Python environment and replicates key QGIS vector operations (filtering, joins, styling) using GeoPandas.
    


    ## Learning goals
    
    - Load vector datasets from the shared data repository into GeoPandas GeoDataFrames.
    - Perform attribute cleaning, filtering, and spatial joins programmatically.
    - Export tidy GeoPackage or GeoJSON outputs for further use in QGIS or reporting.
    


    ## 1. Environment validation
    
    Run the cell below to confirm core packages are available inside the dev container or conda environment.
    


In [ ]:
    import importlib
    import sys
    
    CORE = ["geopandas", "pyproj", "shapely", "matplotlib", "contextily"]
    print(f"Python {sys.version}")
    
    for module_name in CORE:
        try:
            module = importlib.import_module(module_name)
            version = getattr(module, "__version__", "?")
            print(f"✓ {module_name} {version}")
        except ModuleNotFoundError as exc:
            print(f"✗ {module_name}: {exc}")
    


    ## 2. Load datasets
    
    Update the paths once datasets are staged in `data/processed/`. Document source metadata in `resources/docs/data-inventory.md` as you confirm files.
    


In [ ]:
    from pathlib import Path
    
    import geopandas as gpd
    
    DATA_ROOT = Path("..") / "data" / "processed"
    NEIGHBOURHOODS = DATA_ROOT / "week08" / "neighbourhoods.geojson"
    INCIDENTS = DATA_ROOT / "week08" / "incidents.geojson"
    
    if not NEIGHBOURHOODS.exists():
        raise FileNotFoundError("Provide neighbourhood polygons at data/processed/week08/neighbourhoods.geojson")
    if not INCIDENTS.exists():
        raise FileNotFoundError("Provide incident point data at data/processed/week08/incidents.geojson")
    
    neighbourhoods = gpd.read_file(NEIGHBOURHOODS)
    incidents = gpd.read_file(INCIDENTS)
    
    neighbourhoods.head()
    


    ## 3. Cleaning & enrichment
    
    Fill in the TODOs to clean attributes, normalise column names, and join domain-specific context (e.g., census metrics).
    


In [ ]:
    # TODO: Standardise column names and select relevant fields
    neighbourhoods_clean = (
        neighbourhoods.rename(columns=str.lower)
        .assign(area_km2=neighbourhoods.geometry.to_crs(3857).area / 1e6)
    )
    
    incidents_clean = incidents.rename(columns=str.lower)
    print(neighbourhoods_clean.columns)
    print(incidents_clean.columns)
    


    ## 4. Spatial join example
    
    Replicate the QGIS hotspot analysis by joining point incidents to polygons and computing per-area rates.
    


In [ ]:
    joined = gpd.sjoin(incidents_clean, neighbourhoods_clean, predicate="within", how="left")
    incident_counts = (
        joined.groupby("neighbourhood_id").size().rename("incident_count")
    )
    neighbourhoods_summary = neighbourhoods_clean.merge(
        incident_counts, left_on="neighbourhood_id", right_index=True, how="left"
    ).fillna({"incident_count": 0})
    neighbourhoods_summary["rate_per_km2"] = neighbourhoods_summary["incident_count"] / neighbourhoods_summary["area_km2"]
    neighbourhoods_summary.head()
    


    ## 5. Visualise results
    
    Produce a quick classification map; align symbology choices with the styles applied in QGIS during the crime clinic.
    


In [ ]:
    import matplotlib.pyplot as plt
    
    ax = neighbourhoods_summary.plot(
        column="rate_per_km2",
        scheme="Quantiles",
        k=5,
        cmap="YlOrRd",
        legend=True,
        figsize=(10, 6),
    )
    ax.set_title("Incident density per neighbourhood")
    ax.set_axis_off()
    plt.show()
    


    ## 6. Export outputs
    
    Save cleaned layers for reuse in Week 9/10 notebooks or to re-import into QGIS for cartographic refinement.
    


In [ ]:
    OUTPUT_DIR = Path("..") / "data" / "processed" / "week08"
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    
    out_path = OUTPUT_DIR / "neighbourhoods_summary.gpkg"
    neighbourhoods_summary.to_file(out_path, layer="neighbourhoods_summary", driver="GPKG")
    print(f"Exported {out_path}")
    


    ## 7. Reflection
    
    Use the prompts to capture key differences between QGIS and Python workflows. Copy answers into your reflection doc.
    


In [ ]:
    reflection = {
        "automation_win": "What step felt faster/more reproducible in Python?",
        "qgis_alignment": "How will you reflect these outputs back in QGIS layouts?",
        "questions": "List open questions for instructors."
    }
    for key, value in reflection.items():
        print(f"{key}: {value}")
    
